# 00 - Setup and a first look at a public flight

A small notebook that checks the toolchain works before the capability notebooks (`01_frames_and_poses`, `02_georeference`, ...) build on it. It mirrors the [BAMBI Dataset introduction](https://github.com/bambi-eco/Dataset/blob/main/introduction.ipynb): install `bambi-detection` and an alfspy backend, fetch one flight from Zenodo through the Dataset repository's own script, and look at what a flight contains.

Everything runs headless - the same notebook is executed in CI on every release, so if you can read this rendered with outputs, it ran.

## Environment

`ensure_environment()` is idempotent: on a checkout it installs the package editable, on Colab it installs the pinned release tag. It also clones the Dataset repository, whose `download_from_zenodo.py` does the downloading - so a change to the Zenodo layout is fixed in exactly one place.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))       # notebooks/ on the path for _setup
from _setup import ensure_environment, get_flight, find, DATA_DIR

ensure_environment()

## One flight

Flight `146` is the one the Dataset introduction uses - a red-deer flight with the RGB and thermal videos side by side. The `base` version carries the processed video, the MOT ground truth and the poses; it is a few hundred megabytes and cached after the first run.

In [ ]:
flight = get_flight("146", version="base")
for p in sorted(flight.rglob("*"))[:25]:
    if p.is_file():
        print(f"{p.stat().st_size/1e6:8.1f} MB  {p.relative_to(flight)}")

## What a public pose looks like

The `base` download carries the poses **geographically**: WGS84 `lat/lng/alt` plus gimbal `pitch/roll/yaw` per frame. That is deliberately not yet the DEM-local `location/rotation` form the pipeline computes with - turning one into the other needs a DEM and is exactly what `01_frames_and_poses` demonstrates (the Dataset repo's `dem_from_poses.py` + `add_relative_dem_position_to_poses.py` do it standalone).

Here we just read the file the way every downstream step does, and take the numpy view - `(N,3)` geographic positions and `(N,3)` `[pitch, roll, yaw]` - the shape the engine's public functions speak from Phase 1 on.

In [ ]:
import json
import numpy as np

poses_file = find(flight, "*poses*.json")[0]
poses = json.load(open(poses_file, encoding="utf-8"))
images = poses["images"]
print(f"{len(images)} poses in {poses_file.name}")
print("drone/camera:", poses.get("drone"), "/", poses.get("camera"), "| origin:", poses.get("origin"))
print("first pose  :", images[0])

# The numpy view.
lla = np.asarray([[im["lat"], im["lng"], im["alt"]] for im in images], dtype=float)   # (N,3) WGS84
pry = np.asarray([[im["pitch"], im["roll"], im["yaw"]] for im in images], dtype=float)  # (N,3) degrees
print("lla", lla.shape, "pry", pry.shape)
print("lat/lng span (deg):", np.ptp(lla[:, :2], axis=0).round(5),
      "| altitude (m):", lla[:, 2].min().round(1), "..", lla[:, 2].max().round(1))
print("gimbal pitch range (deg):", pry[:, 0].min().round(1), "..", pry[:, 0].max().round(1),
      "  (near 360 / 0 = pointing straight down)")

## Which alfspy backend is installed?

`bambi.util.render_context` answers that without the caller having to care - it is the seam that lets the same notebook run on the ModernGL build or the PyTorch build.

In [ ]:
from bambi.util.render_context import render_backend, make_render_context

print("alfspy backend:", render_backend())
ctx = make_render_context()
print("render context:", type(ctx).__name__, "| device:", getattr(ctx, "device", "n/a"))

## Where the flight went

Look at the flight path from above - the same plot the QGIS plugin draws as its flight-route layer, without QGIS.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(lla[:, 1], lla[:, 0], c=lla[:, 2], s=4, cmap="viridis")
ax.plot(lla[:, 1], lla[:, 0], lw=0.5, alpha=0.5)
ax.set_aspect(1 / np.cos(np.radians(lla[:, 0].mean())))   # metres look square at this latitude
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"flight 146 - {len(lla)} poses")
plt.colorbar(sc, label="altitude (m)")
plt.show()

## Next

- `01_frames_and_poses` - from the raw DJI video to undistorted frames and a poses file, with the calibration guard catching a wrong preset
- `02_georeference` - MOT labels to world coordinates on the DEM, nadir vs oblique
- `03_tracking`, `04_survey_analytics`, `05_rendering`

Each lands with the code it demonstrates.